In [ ]:
import os
import requests
import pandas as pd
import concurrent.futures

data_dir = os.path.abspath("/srv/data1/general/immunopeptides_data/")
protein_protein_files = os.path.join(data_dir, 'databases/benchmark_data/output_data/peptide-like-proteins.csv')
protein_ligand_files = os.path.join(data_dir, 'databases/benchmark_data/output_data/peptide_like_ligands.csv')

# Merge the two files
protein_protein_df = pd.read_csv(protein_protein_files)
protein_protein_df['source'] = 'protein_protein'

protein_ligand_df = pd.read_csv(protein_ligand_files)
protein_ligand_df['source'] = 'protein_ligand'

peptides_df = pd.concat([protein_protein_df, protein_ligand_df])
peptides_df.head()

                                         Ligand name     Type PDB code  \
0  2cpk.pdf (20-mer) a 20-amino acid substrate an...  Peptide     2cpk   
1            1ppe.pdf (29-mer) CMTI-squash inhibitor  Peptide     1ppe   
2      1smf.pdf (22-mer) incomplete ligand structure  Peptide     1smf   
3  1ihs.pdf (21-mer) hirutonin-2 with human a-thr...  Peptide     1ihs   
4  1dit.pdf (20-mer) A DIVALENT PEPTIDE INHIBITOR...  Peptide     1dit   

  Resolution  Release year Binding data Reference           source  
0       2.70          1993     Ki=2.3nM  2cpk.pdf  protein_protein  
1       2.00          1994       Kd=3pM  1ppe.pdf  protein_protein  
2       2.10          1994    Ki=0.12uM  1smf.pdf  protein_protein  
3       2.00          1994     Ki=0.3nM  1ihs.pdf  protein_protein  
4       2.30          1996       Ki=1pM  1dit.pdf  protein_protein  


In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor

# Protein PDBs
# input_file = os.path.join(base_dir, "data/databases/benchmark_data/output_data/peptide-like-proteins.csv")
output_dir = os.path.join(data_dir, "databases/benchmark_data/output_data/peptide-pdb-files")

df = peptides_df
# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

def download_pdb(pdb_id, output_dir):
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    response = requests.get(url)
    if response.status_code == 200:
        with open(os.path.join(output_dir, f'{pdb_id}.pdb'), 'wb') as file:
            file.write(response.content)
        print(f'Successfully downloaded {pdb_id}.pdb')
    else:
        print(f'Failed to download {pdb_id}.pdb')

def download_pdbs_in_batch(pdb_ids, output_dir, max_workers=5):
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        executor.map(lambda pdb_id: download_pdb(pdb_id, output_dir), pdb_ids)

# Uncomment the line below to download the PDB files | comment it out if you have already downloaded the PDB files to av
download_pdbs_in_batch(df['PDB code'], output_dir, max_workers=5)

Successfully downloaded 1smf.pdb
Successfully downloaded 1ppe.pdb
Successfully downloaded 1dit.pdb
Successfully downloaded 2cpk.pdb
Successfully downloaded 1ihs.pdb
Successfully downloaded 1an1.pdb
Successfully downloaded 1a3b.pdb
Successfully downloaded 1fmo.pdb
Successfully downloaded 1hia.pdb
Successfully downloaded 1ldt.pdb
Successfully downloaded 1vrk.pdb
Successfully downloaded 1g9i.pdb
Successfully downloaded 1qur.pdb
Successfully downloaded 1vpp.pdb
Successfully downloaded 4thn.pdb
Successfully downloaded 1dpj.pdb
Successfully downloaded 1clv.pdb
Successfully downloaded 1i5k.pdb
Successfully downloaded 1g5j.pdb
Successfully downloaded 1ees.pdb
Successfully downloaded 1g0v.pdb
Successfully downloaded 1gl0.pdb
Successfully downloaded 1lj2.pdb
Successfully downloaded 1lm8.pdb
Successfully downloaded 1lqb.pdb
Successfully downloaded 1gl1.pdb
Successfully downloaded 1k2d.pdb
Successfully downloaded 1gng.pdb
Successfully downloaded 1kbh.pdb
Successfully downloaded 1j2j.pdb
Successful


### Checks if complexes are dual chained, that peptides are of correct length (< 40) and that that they are unmodified.

TODO: 
- [x] (possibly) add additional checks for the complexes. 
- [x] determine if we require additional data as we are getting 200 complexes atm.



In [27]:


from Bio import PDB
from Bio.PDB.Polypeptide import is_aa

peptide_pdb = os.path.join(base_dir, "data/databases/benchmark_data/output_data/peptide-pdb-files")

def has_peptide_and_protein(pdb_file, peptide_max_length=40):
    """
    Checks whether the PDB file contains exactly two polypeptide chains:
    one chain with fewer than `peptide_max_length` amino acids (a "peptide"),
    and one chain with >= `peptide_max_length` amino acids (a "protein").

    Returns a tuple (bool, str) where the bool indicates if the condition is met,
    and the str indicates the reason for exclusion if not met.
    """

    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure("temp_struct", pdb_file)

    chain_residue_counts = []
    model = structure[0]

    for chain in model:
        count_aa = 0
        for residue in chain.get_residues():
            if is_aa(residue, standard=True):
                count_aa += 1
        if count_aa > 0:
            chain_residue_counts.append(count_aa)

    if len(chain_residue_counts) != 2:
        return False, "not_two_chains"

    chain_residue_counts.sort()
    if chain_residue_counts[0] >= peptide_max_length:
        return False, "no_peptide"
    if chain_residue_counts[1] < peptide_max_length:
        return False, "no_protein"

    return True, "meets_condition"

if __name__ == "__main__":
    """
    Run in root.
    """
    pdb_ids = df['PDB code'].tolist()
    true_count = 0
    total_count = 0
    exclusion_counts = {"not_two_chains": 0, "no_peptide": 0, "no_protein": 0}

    for pdb_id in pdb_ids:
        pdb_file = os.path.join(peptide_pdb, f'{pdb_id}.pdb')
        if os.path.exists(pdb_file):
            result, reason = has_peptide_and_protein(pdb_file)
            print(f'{pdb_id}: {result} ({reason})')
            total_count += 1
            if result:
                true_count += 1
            else:
                exclusion_counts[reason] += 1

    print(f'Number of PDB files with the specified condition: {true_count}')
    print(f'Total number of PDB files checked: {total_count}')
    print(f'Exclusion counts: {exclusion_counts}')

2cpk: True (meets_condition)
1ppe: True (meets_condition)
1smf: True (meets_condition)
1ihs: False (not_two_chains)
1dit: False (not_two_chains)
1hia: False (not_two_chains)
1a3b: False (not_two_chains)
1an1: False (no_peptide)
1ldt: False (no_peptide)
1fmo: True (meets_condition)
1vpp: False (not_two_chains)
1vrk: True (meets_condition)
4thn: False (not_two_chains)
1qur: False (not_two_chains)
1g9i: True (meets_condition)
1ees: False (no_peptide)
1clv: True (meets_condition)
1dpj: True (meets_condition)
1g5j: True (meets_condition)
1i5k: False (not_two_chains)
1g0v: True (meets_condition)
1gl1: False (not_two_chains)
1gl0: True (meets_condition)
1lm8: False (not_two_chains)
1lj2: False (not_two_chains)
1lqb: False (not_two_chains)
1kbh: False (no_peptide)
1gng: False (not_two_chains)
1k2d: False (not_two_chains)
1jgn: True (meets_condition)
1j2j: False (no_peptide)
1jh4: True (meets_condition)
1mzw: True (meets_condition)
1pjm: True (meets_condition)
1o9a: True (meets_condition)
1m5n:

KeyboardInterrupt: 